# GridPulse UK — Historical Carbon-Intensity Ingestion

This notebook performs a historical backfill from the NESO Carbon Intensity API.

## Objectives

1. Request a defined historical date range.
2. Validate the API response before storing it.
3. Preserve the response as immutable raw JSON.
4. Transform historical periods into Bronze Delta records.
5. Make the historical load safe to rerun.

In [0]:
START_DATETIME = "2026-09-01T00:00Z"
END_DATETIME = "2026-09-08T00:00Z"

BASE_URL = "https://api.carbonintensity.org.uk"

HISTORICAL_API_URL = (
    f"{BASE_URL}/intensity/"
    f"{START_DATETIME}/{END_DATETIME}"
)

print(f"Historical API URL: {HISTORICAL_API_URL}")

In [0]:
import requests

response = requests.get(
    HISTORICAL_API_URL,
    timeout=60
)

print(f"HTTP status: {response.status_code}")

response.raise_for_status()

historical_payload = response.json()

print(f"Top-level keys: {list(historical_payload.keys())}")
print(f"Historical periods returned: {len(historical_payload['data'])}")

In [0]:
first_three_periods = historical_payload["data"][:3]
last_three_periods = historical_payload["data"][-3:]

print("First three periods:")
display(first_three_periods)

print("Last three periods:")
display(last_three_periods)

In [0]:
period_count = len(historical_payload["data"])

assert response.status_code == 200, (
    "The historical API request was unsuccessful."
)

assert "data" in historical_payload, (
    "The API response does not contain a data field."
)

assert period_count > 0, (
    "The historical API returned no electricity periods."
)

print(
    f"Historical API validation passed "
    f"with {period_count} periods."
)

In [0]:
from datetime import datetime, timezone
import json

ingested_at = datetime.now(timezone.utc)

raw_envelope = {
    "source_system": "neso_carbon_intensity_api",
    "dataset": "national_historical_carbon_intensity",
    "source_url": HISTORICAL_API_URL,
    "request_start_utc": START_DATETIME,
    "request_end_utc": END_DATETIME,
    "ingested_at_utc": ingested_at.isoformat(),
    "record_count": period_count,
    "payload": historical_payload
}

print(f"Source system: {raw_envelope['source_system']}")
print(f"Dataset: {raw_envelope['dataset']}")
print(f"Requested records: {raw_envelope['record_count']}")
print(f"Ingested at: {raw_envelope['ingested_at_utc']}")

In [0]:
ingestion_date_path = ingested_at.strftime("%Y/%m/%d")
file_timestamp = ingested_at.strftime("%Y%m%dT%H%M%SZ")

range_start_label = START_DATETIME[:10].replace("-", "")
range_end_label = END_DATETIME[:10].replace("-", "")

historical_landing_directory = (
    "/Volumes/gridpulse/bronze/raw_landing/"
    f"carbon_intensity/national_historical/{ingestion_date_path}"
)

historical_file_path = (
    f"{historical_landing_directory}/"
    f"carbon_intensity_"
    f"{range_start_label}_"
    f"{range_end_label}_"
    f"{file_timestamp}.json"
)

print(f"Landing directory: {historical_landing_directory}")
print(f"File path: {historical_file_path}")

In [0]:
dbutils.fs.mkdirs(historical_landing_directory)

dbutils.fs.put(
    historical_file_path,
    json.dumps(raw_envelope),
    overwrite=False
)

print(
    "Historical raw file written successfully:\n"
    f"{historical_file_path}"
)

In [0]:
historical_files = dbutils.fs.ls(
    historical_landing_directory
)

display(historical_files)

In [0]:
historical_raw_test_df = (
    spark.read
    .format("json")
    .load(historical_file_path)
)

print(
    f"Raw envelope rows read by Spark: "
    f"{historical_raw_test_df.count()}"
)

historical_raw_test_df.printSchema()

In [0]:
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    StringType,
    StructField,
    StructType
)

intensity_schema = StructType([
    StructField("forecast", IntegerType(), True),
    StructField("actual", IntegerType(), True),
    StructField("index", StringType(), True)
])

period_schema = StructType([
    StructField("from", StringType(), True),
    StructField("to", StringType(), True),
    StructField("intensity", intensity_schema, True)
])

payload_schema = StructType([
    StructField(
        "data",
        ArrayType(period_schema),
        True
    )
])

historical_envelope_schema = StructType([
    StructField("source_system", StringType(), False),
    StructField("dataset", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("request_start_utc", StringType(), False),
    StructField("request_end_utc", StringType(), False),
    StructField("ingested_at_utc", StringType(), False),
    StructField("record_count", IntegerType(), False),
    StructField("payload", payload_schema, True)
])

In [0]:
HISTORICAL_RAW_ROOT = (
    "/Volumes/gridpulse/bronze/raw_landing/"
    "carbon_intensity/national_historical/"
)

BRONZE_TABLE = "gridpulse.bronze.carbon_intensity"

In [0]:
historical_raw_df = (
    spark.read
    .format("json")
    .schema(historical_envelope_schema)
    .option("recursiveFileLookup", "true")
    .load(HISTORICAL_RAW_ROOT)
)

print(
    f"Historical envelopes found: "
    f"{historical_raw_df.count()}"
)

historical_raw_df.printSchema()

In [0]:
from pyspark.sql.functions import (
    col,
    explode_outer
)

historical_exploded_df = (
    historical_raw_df
    .withColumn(
        "period",
        explode_outer(col("payload.data"))
    )
)

print(
    f"Historical periods after explode: "
    f"{historical_exploded_df.count()}"
)

In [0]:
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    sha2,
    to_timestamp
)

historical_bronze_df = (
    historical_exploded_df
    .select(
        col("source_system"),

        to_timestamp(
            col("ingested_at_utc")
        ).alias("ingested_at_utc"),

        to_timestamp(
            col("period.from")
        ).alias("period_start_utc"),

        to_timestamp(
            col("period.to")
        ).alias("period_end_utc"),

        col("period.intensity.forecast")
            .alias("forecast_gco2_per_kwh"),

        col("period.intensity.actual")
            .alias("actual_gco2_per_kwh"),

        col("period.intensity.index")
            .alias("intensity_index")
    )
)

In [0]:
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    sha2,
    to_timestamp
)

historical_bronze_ready_df = (
    historical_exploded_df
    .select(
        col("source_system"),
        to_timestamp(col("ingested_at_utc")).alias("ingested_at_utc"),
        to_timestamp(col("period.from"), "yyyy-MM-dd'T'HH:mm'Z'").alias("period_start_utc"),
        to_timestamp(col("period.to"), "yyyy-MM-dd'T'HH:mm'Z'").alias("period_end_utc"),
        col("period.intensity.forecast").alias("forecast_gco2_per_kwh"),
        col("period.intensity.actual").alias("actual_gco2_per_kwh"),
        col("period.intensity.index").alias("intensity_index")
    )
    .withColumn(
        "record_id",
        sha2(
            concat_ws(
                "||",
                col("source_system"),
                col("ingested_at_utc").cast("string"),
                col("period_start_utc").cast("string"),
                col("period_end_utc").cast("string")
            ),
            256
        )
    )
    .withColumn(
        "loaded_at_utc",
        current_timestamp()
    )
    .select(
        "record_id",
        "source_system",
        "ingested_at_utc",
        "period_start_utc",
        "period_end_utc",
        "forecast_gco2_per_kwh",
        "actual_gco2_per_kwh",
        "intensity_index",
        "loaded_at_utc"
    )
)

display(
    historical_bronze_ready_df
    .orderBy("period_start_utc")
    .limit(10)
)

In [0]:
historical_record_count = (
    historical_bronze_ready_df.count()
)

missing_record_ids = (
    historical_bronze_ready_df
    .filter(col("record_id").isNull())
    .count()
)

missing_event_times = (
    historical_bronze_ready_df
    .filter(
        col("period_start_utc").isNull()
        | col("period_end_utc").isNull()
    )
    .count()
)

invalid_period_order = (
    historical_bronze_ready_df
    .filter(
        col("period_end_utc")
        <= col("period_start_utc")
    )
    .count()
)

print(
    f"Historical records: {historical_record_count}"
)
print(
    f"Missing record IDs: {missing_record_ids}"
)
print(
    f"Missing event times: {missing_event_times}"
)
print(
    f"Invalid period order: {invalid_period_order}"
)

In [0]:
assert historical_record_count > 0, (
    "No historical periods were extracted."
)

assert missing_record_ids == 0, (
    "Some historical records have no record_id."
)

assert missing_event_times == 0, (
    "Some historical records have missing event times."
)

assert invalid_period_order == 0, (
    "Some periods end before or at their start time."
)

print("Historical Bronze validation passed.")

In [0]:
distinct_record_count = (
    historical_bronze_ready_df
    .select("record_id")
    .distinct()
    .count()
)

duplicate_record_count = (
    historical_record_count
    - distinct_record_count
)

print(
    f"Distinct record IDs: {distinct_record_count}"
)
print(
    f"Duplicate record IDs: {duplicate_record_count}"
)

assert duplicate_record_count == 0, (
    "The incoming historical batch contains duplicate record IDs."
)

In [0]:
%sql
SELECT COUNT(*) AS count_before_historical_load
FROM gridpulse.bronze.carbon_intensity;

In [0]:
from delta.tables import DeltaTable

bronze_delta_table = DeltaTable.forName(
    spark,
    BRONZE_TABLE
)

(
    bronze_delta_table.alias("target")
    .merge(
        historical_bronze_ready_df.alias("source"),
        "target.record_id = source.record_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    f"Historical records merged into: "
    f"{BRONZE_TABLE}"
)

In [0]:
%sql
SELECT COUNT(*) AS count_after_historical_load
FROM gridpulse.bronze.carbon_intensity;

In [0]:
%sql
SELECT
    MIN(period_start_utc) AS earliest_period,
    MAX(period_end_utc) AS latest_period,
    COUNT(*) AS total_records
FROM gridpulse.bronze.carbon_intensity;

In [0]:
%sql
SELECT
    DATE(period_start_utc) AS electricity_date,
    COUNT(*) AS period_count,
    ROUND(
        AVG(actual_gco2_per_kwh),
        2
    ) AS average_actual_intensity
FROM gridpulse.bronze.carbon_intensity
GROUP BY DATE(period_start_utc)
ORDER BY electricity_date;

In [0]:
%sql
SELECT COUNT(*) AS total_bronze_records
FROM gridpulse.bronze.carbon_intensity;